# Research extension Q5: stochastic softmax placement

**Question:** Modify EVA so that additional installations are assigned stochastically using the softmax of ZIP scores, rather than filling ZIPs in descending score order. Report the results.

This experiment extends our **Q4 methodological reconstruction**, not a verified numerical replication of the paper. It preserves Q4's data, min-max input normalization, six network inputs, genetic operators, truncated lexicase selection, four objectives, baseline and installation-unit conventions. We use 10,243 eligible ZIPs and a final budget of 2,000,000 additional installation units. Capacity is `floor(count_qualified)`. The energy and carbon quantities remain Q4's toolkit indicators; their ratios are dimensionless, not engineering forecasts. Equity measures use existing plus additional installations and the same median-based groups as Q4.

## Placement rule

Let $s_i$ be the network's raw score for ZIP $i$, $u_i$ its additional-installation capacity and $a_i(t)$ the units assigned so far. Define $A_t=\{i:a_i(t)<u_i\}$ and $m_t=\max_{j\in A_t}s_j$. The next unit is assigned with probability

$$
p_i(t)=\begin{cases}\displaystyle\frac{\exp((s_i-m_t)/T)}{\sum_{j\in A_t}\exp((s_j-m_t)/T)},&i\in A_t,\\0,&i\notin A_t.\end{cases}
$$

The subtraction of $m_t$ improves numerical stability without changing probabilities. $T>0$ is temperature. These are probabilities over eligible ZIPs, **not** over remaining rooftops. A full ZIP is removed and probabilities are renormalized. We sample integer counts and enforce both the exact budget and every capacity.

As $T\to0$, the highest scores receive almost all probability; distinct-score placement approaches greedy filling. Equal maximum scores remain stochastic. As $T\to\infty$, choices become uniform over currently nonfull ZIPs. This does not imply uniform final allocations, population coverage or demographic equity.

At any given step this probability vector solves

$$
\max_{p\in\Delta(A_t)}\left[\sum_i p_i s_i+T H(p)\right],\qquad H(p)=-\sum_i p_i\log p_i.
$$

Thus temperature trades preference for high scores against randomness in ZIP selection. This identity explains the local sampling rule; it is not an additional fifth EVA objective or a proof of globally optimal deployment.

## Efficient capacity handling

A multinomial draw gives the counts of independent categorical proposals. Propose the number of units still needed, accept at most each ZIP's remaining capacity, and redraw the rejected count over nonfull ZIPs. This is batched rejection sampling. For a proposal batch, the number accepted at a ZIP is the smaller of its proposal count and remaining capacity, regardless of proposal order. The batch contains no more proposals than units still needed. Conditioning subsequent accepted draws on the remaining ZIPs gives exactly the sequential softmax law, subject to floating-point underflow at extreme score differences. Recomputing the maximum after saturation allows lower-score ZIPs to become selectable.

This avoids a Python loop over two million units. Simply clipping one multinomial draw and stopping would lose units. Arbitrarily distributing the clipped excess would change the policy.

## Why the score scale matters

Greedy placement is invariant under every strictly increasing score transformation. Softmax is invariant to a constant shift but sensitive to scaling:

$$
\operatorname{softmax}(c s/T)=\operatorname{softmax}(s/(T/c)),\qquad c>0.
$$

Therefore scores that are equivalent for Q4 need not define equivalent stochastic policies. We use **raw outputs**, without converting them to percentiles or rescaling their range. The main illustrative setting is $T=0.01$; it is not claimed to be an optimal temperature. A separate fixed-network temperature sweep shows how strongly this choice interacts with calibration. Retraining may change score spread even when output activation remains sigmoid.

## Experiment and fitness

For each evolutionary seed 0, 1 and 2, compare (a) Q4's saved network with greedy placement, (b) that same network with softmax placement, and (c) EVA retrained using softmax placement in candidate evaluation. This separates the effect of changing deployment from the effect of adapting the learned score function.

Retraining evaluates every candidate on placement seeds 10, 11, 12 and 13. For objective $k$, use

$$
\widehat F_k(\theta)=\frac14\sum_{r=1}^4 F_k(A_{\theta,r}),\qquad R_k(\theta)=\frac{\widehat F_k(\theta)}{F_k(A_{\mathrm{baseline}})}.
$$

The four objectives remain separate in truncated lexicase selection; the sum of ratios (CPR) retains Q4's survivor ranking and checkpoint convention. Population size is 30, with six generations per stage. Adaptive selectivity tuning allows at most six stages and returns the first model whose four estimated ratios reach one, or the best estimated-CPR model if this criterion is never met. The latter outcome must be reported as a failure to meet the criterion. Runs needing more stages use more compute, so this is not an equal-compute comparison with Q4.

Average **objective values of realized allocations**, not the objective of the average allocation. Carbon and energy are linear, but the equity score is $1-|D|/(I_0+n)$ with fixed denominator. Consequently,

$$
\mathbb E[F_{\mathrm{equity}}(A)]\leq F_{\mathrm{equity}}(\mathbb E[A]).
$$

For example, one additional unit randomly split between two empty groups gives equity zero in each realization, while the fractional average allocation $(0.5,0.5)$ has equity one. Averaging allocations would hide realized imbalance.

Report 100 held-out deployment draws using seeds 20000 through 20099, separate from the training pool. Primary plots use evolutionary seed 0, chosen before comparing results. Deployment paths use another 100 seeds, 30000 through 30099, carrying forward installed units at checkpoints from 100,000 through 2,000,000. No installations move between checkpoints. Plot bands are one deployment standard deviation conditional on a fixed network, not confidence intervals or estimates of evolutionary or data uncertainty.


Install dependencies once with `pip install -r requirements.txt`, then run from this folder. The pinned SunSight toolkit (`../SunSight_Q3`), shared inputs (`../Data`) and shared helpers (`../Utils`) ship with the repository, so no git or internet access is needed. Saved Q4 controls are read from `../Q4_EVA_reconstruction/Models`; all new softmax training runs are executed below. The full new implementation is expanded in the next code cells.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "Utils"))
from paths import DATA, NEAT_CONFIG, Q4

root = Path.cwd()
models, results, plots = root / "Models", root / "Results", root / "Plots"

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from q4_eva import prepare_experiment

experiment = prepare_experiment(pd.read_csv(DATA / "data_by_zip.csv"), pd.read_csv(DATA / "Q4_EIA_March_2024_2025.csv"))
temperature = 0.01
evolution_seeds = [0, 1, 2]
training_seeds = (10, 11, 12, 13)
test_seeds = range(20_000, 20_100)
q4_models = {seed: pickle.loads((Q4 / "Models" / f"eva_seed_{seed}.pkl").read_bytes()) for seed in evolution_seeds}


In [ ]:
import copy
import itertools
import random

import neat
import numpy as np
import pandas as pd

from q4_eva import OBJECTIVES, baseline_allocation, evaluate, network_scores, rank_allocation


def softmax_probabilities(scores, temperature):
    scores = np.asarray(scores, dtype=float)
    if scores.ndim != 1 or len(scores) == 0 or not np.isfinite(scores).all():
        raise ValueError("Scores must be a nonempty finite one-dimensional array.")
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError("Temperature must be finite and positive.")
    probabilities = np.exp((scores - scores.max()) / temperature)
    return probabilities / probabilities.sum()


def softmax_allocation(scores, capacities, n, temperature, rng):
    scores = np.asarray(scores, dtype=float)
    capacities = np.asarray(capacities)
    if scores.ndim != 1 or capacities.shape != scores.shape or not np.isfinite(scores).all():
        raise ValueError("Scores and capacities must be matching finite one-dimensional arrays.")
    if not np.isfinite(capacities).all() or (capacities < 0).any() or (capacities != np.floor(capacities)).any():
        raise ValueError("Capacities must be nonnegative integers.")
    if not isinstance(n, (int, np.integer)) or n < 0 or n > capacities.sum():
        raise ValueError("n must be a feasible nonnegative integer.")
    if not np.isfinite(temperature) or temperature <= 0:
        raise ValueError("Temperature must be finite and positive.")
    capacities = capacities.astype(np.int64)
    allocation = np.zeros(len(scores), dtype=np.int64)
    remaining = int(n)
    while remaining:
        active = np.flatnonzero(allocation < capacities)
        probabilities = softmax_probabilities(scores[active], temperature)
        proposals = rng.multinomial(remaining, probabilities)
        accepted = np.minimum(proposals, capacities[active] - allocation[active])
        allocation[active] += accepted
        remaining -= int(accepted.sum())
    return allocation


def mean_objectives(experiment, scores, n, temperature, seeds):
    values = []
    for seed in seeds:
        allocation = softmax_allocation(scores, experiment["capacity"], n, temperature, np.random.default_rng(seed))
        values.append(evaluate(experiment, allocation))
    return np.mean(values, axis=0)


def train_policy(experiment, config_path, seed, temperature=None, training_seeds=(10, 11, 12, 13), generations=6, n=2_000_000, selectivity=None):
    random.seed(seed)
    np.random.seed(seed)
    config = neat.Config(neat.DefaultGenome, neat.DefaultReproduction, neat.DefaultSpeciesSet, neat.DefaultStagnation, str(config_path))
    baseline = evaluate(experiment, baseline_allocation(experiment, n))
    selectivity = np.ones(4) if selectivity is None else np.asarray(selectivity, dtype=float)
    population = []
    next_id = 0
    for _ in range(30):
        genome = neat.DefaultGenome(next_id)
        next_id += 1
        genome.configure_new(config.genome_config)
        population.append(genome)
    best_genome = None
    best_cpr = -np.inf
    history = []
    for generation in range(generations):
        ratios = []
        for genome in population:
            network = neat.nn.FeedForwardNetwork.create(genome, config)
            scores = network_scores(network, experiment["inputs"])
            if temperature is None:
                values = evaluate(experiment, rank_allocation(scores, experiment["capacity"], n))
            else:
                values = mean_objectives(experiment, scores, n, temperature, training_seeds)
            ratios.append(values / baseline)
        ratios = np.array(ratios)
        cpr = ratios.sum(axis=1)
        for genome, fitness in zip(population, cpr):
            genome.fitness = float(fitness)
        survivors = np.arange(len(population))
        for objective in np.random.permutation(4):
            survivors = survivors[np.argsort(-ratios[survivors, objective], kind="stable")]
            keep = int(np.ceil(len(survivors) * 0.3 ** (selectivity[objective] / selectivity.sum())))
            survivors = survivors[:keep]
        survivors = survivors[np.argsort(-cpr[survivors], kind="stable")][:9]
        winner = int(survivors[0])
        history.append([generation, *ratios[winner], cpr[winner]])
        if cpr[winner] > best_cpr:
            best_cpr = float(cpr[winner])
            best_genome = copy.deepcopy(population[winner])
        if generation + 1 < generations:
            selected = [population[i] for i in survivors]
            pairs = list(itertools.combinations(selected, 2))
            random.shuffle(pairs)
            new_population = [copy.deepcopy(selected[0])]
            for first, second in pairs[:29]:
                child = neat.DefaultGenome(next_id)
                next_id += 1
                child.configure_crossover(first, second, config.genome_config)
                child.mutate(config.genome_config)
                new_population.append(child)
            population = new_population
    network = neat.nn.FeedForwardNetwork.create(best_genome, config)
    return network, pd.DataFrame(history, columns=["Generation", *OBJECTIVES, "CPR"])


def train_softmax_with_tuning(experiment, config_path, seed=0, temperature=0.01, training_seeds=(10, 11, 12, 13), max_rounds=6):
    selectivity = np.ones(4)
    baseline = evaluate(experiment, baseline_allocation(experiment, 2_000_000))
    best_network = None
    best_history = None
    best_cpr = -np.inf
    rows = []
    columns = ["Stage", *["Selectivity: " + name for name in OBJECTIVES], *OBJECTIVES]
    for stage in range(max_rounds):
        network, history = train_policy(experiment, config_path, seed + 1000 * stage, temperature, training_seeds, selectivity=selectivity)
        scores = network_scores(network, experiment["inputs"])
        ratios = mean_objectives(experiment, scores, 2_000_000, temperature, training_seeds) / baseline
        rows.append([stage, *selectivity, *ratios])
        if ratios.sum() > best_cpr:
            best_network = network
            best_history = history
            best_cpr = float(ratios.sum())
        if (ratios >= 1).all():
            return network, pd.DataFrame(rows, columns=columns), history
        selectivity = np.where(ratios < 1, selectivity / np.maximum(ratios, 1e-9), selectivity)
    return best_network, pd.DataFrame(rows, columns=columns), best_history


def softmax_path(scores, capacities, budgets, temperature, rng):
    budgets = np.asarray(budgets)
    if budgets.ndim != 1 or (budgets < 0).any() or (budgets != np.floor(budgets)).any() or (np.diff(budgets) < 0).any():
        raise ValueError("Budgets must be ordered nonnegative integers.")
    if len(budgets) and budgets[-1] > np.sum(capacities):
        raise ValueError("The final budget exceeds capacity.")
    allocation = np.zeros(len(scores), dtype=np.int64)
    path = []
    previous = 0
    for n in budgets:
        allocation += softmax_allocation(scores, capacities - allocation, int(n - previous), temperature, rng)
        path.append(allocation.copy())
        previous = int(n)
    return path


def allocation_diagnostics(allocation):
    shares = allocation[allocation > 0] / allocation.sum()
    return {"ZIPs used": int(len(shares)), "Effective allocated ZIPs": float(np.exp(-np.sum(shares * np.log(shares)))), "Largest ZIP share (%)": float(100 * shares.max())}


def evaluate_policy(experiment, network, temperature, seeds, n=2_000_000):
    scores = network_scores(network, experiment["inputs"])
    baseline = evaluate(experiment, baseline_allocation(experiment, n))
    rows = []
    for seed in seeds:
        allocation = rank_allocation(scores, experiment["capacity"], n) if temperature is None else softmax_allocation(scores, experiment["capacity"], n, temperature, np.random.default_rng(seed))
        ratios = evaluate(experiment, allocation) / baseline
        rows.append({"Deployment seed": int(seed), **dict(zip(OBJECTIVES, ratios)), **allocation_diagnostics(allocation)})
    return pd.DataFrame(rows)


In [ ]:
softmax_models = {}
training_rows = []
for seed in evolution_seeds:
    network, tuning, history = train_softmax_with_tuning(experiment, NEAT_CONFIG, seed, temperature, training_seeds, max_rounds=6)
    softmax_models[seed] = network
    (models / f"softmax_seed_{seed}.pkl").write_bytes(pickle.dumps(network))
    tuning.to_csv(results / f"tuning_seed_{seed}.csv", index=False)
    history.to_csv(results / f"training_seed_{seed}.csv", index=False)
    passed = tuning[OBJECTIVES].ge(1).all(axis=1)
    selected = int(tuning.index[passed][0]) if passed.any() else int(tuning[OBJECTIVES].sum(axis=1).idxmax())
    training_rows.append({"Evolution seed": seed, "Tuning rounds": len(tuning), "Selected stage": selected, "Training criterion met": bool(passed.any())})
training_status = pd.DataFrame(training_rows)
training_status.to_csv(results / "training_status.csv", index=False)
print(training_status.to_string(index=False))


In [ ]:
rows = []
for seed in evolution_seeds:
    policies = [("Q4 deterministic", q4_models[seed], None), ("Q4 network + softmax", q4_models[seed], temperature), ("Retrained softmax EVA", softmax_models[seed], temperature)]
    for label, network, policy_temperature in policies:
        samples = evaluate_policy(experiment, network, policy_temperature, test_seeds if policy_temperature is not None else [20_000])
        samples["Evolution seed"] = seed
        samples["Policy"] = label
        rows.append(samples)
endpoint_samples = pd.concat(rows, ignore_index=True)
metrics = OBJECTIVES + ["ZIPs used", "Effective allocated ZIPs", "Largest ZIP share (%)"]
means = endpoint_samples.groupby(["Evolution seed", "Policy"])[metrics].mean()
sds = endpoint_samples.groupby(["Evolution seed", "Policy"])[metrics].std().fillna(0)
endpoint_samples.to_csv(results / "endpoint_samples.csv", index=False)
means.to_csv(results / "endpoint_means.csv")
sds.to_csv(results / "endpoint_deployment_sd.csv")
print(means.round(6).to_string())
print(sds[OBJECTIVES].round(6).to_string())


In [ ]:
rows = []
for t in [1e-10, 1e-8, 1e-6, 1e-4, 0.01, 0.1, 1.0]:
    samples = evaluate_policy(experiment, q4_models[0], t, test_seeds)
    samples["Temperature"] = t
    rows.append(samples)
temperature_samples = pd.concat(rows, ignore_index=True)
temperature_samples.to_csv(results / "temperature_samples.csv", index=False)
summary = temperature_samples.groupby("Temperature")[OBJECTIVES + ["ZIPs used"]].mean()
fig, axes = plt.subplots(1, 2, figsize=(12, 4), layout="constrained")
for objective in OBJECTIVES:
    axes[0].semilogx(summary.index, summary[objective], marker="o", label=objective)
axes[0].axhline(1, color="black", linestyle=":")
axes[0].set(xlabel="Temperature", ylabel="Mean ratio to reconstructed status quo")
axes[0].legend()
axes[1].semilogx(summary.index, summary["ZIPs used"], marker="o")
axes[1].axhline(289, color="gray", linestyle="--")
axes[1].set(xlabel="Temperature", ylabel="Mean ZIPs receiving units")
fig.savefig(plots / "temperature_sensitivity.png", dpi=180)
plt.show()
plt.close(fig)


In [ ]:
budgets = np.arange(100_000, 2_000_001, 100_000)
scores = network_scores(softmax_models[0], experiment["inputs"])
baselines = {int(n): evaluate(experiment, baseline_allocation(experiment, int(n))) for n in budgets}
rows = []
for seed in range(30_000, 30_100):
    path = softmax_path(scores, experiment["capacity"], budgets, temperature, np.random.default_rng(seed))
    previous = np.zeros(len(scores), dtype=np.int64)
    for n, allocation in zip(budgets, path):
        assert allocation.sum() == n and (allocation <= experiment["capacity"]).all() and (allocation >= previous).all()
        previous = allocation
        rows.append([seed, int(n), *evaluate(experiment, allocation) / baselines[int(n)]])
path_samples = pd.DataFrame(rows, columns=["Deployment seed", "Installations", *OBJECTIVES])
path_samples.to_csv(results / "nested_path_samples.csv", index=False)
path_means = path_samples.groupby("Installations")[OBJECTIVES].mean()
path_sd = path_samples.groupby("Installations")[OBJECTIVES].std()
greedy = np.array([evaluate(experiment, rank_allocation(network_scores(q4_models[0], experiment["inputs"]), experiment["capacity"], int(n))) / baselines[int(n)] for n in budgets])
fig, axes = plt.subplots(2, 2, figsize=(11, 7), layout="constrained")
for k, (ax, objective) in enumerate(zip(axes.flat, OBJECTIVES)):
    x = budgets / 1e6
    y = path_means[objective].to_numpy()
    sd = path_sd[objective].to_numpy()
    ax.plot(x, greedy[:, k], linestyle="--", label="Q4 greedy")
    ax.plot(x, y, label="Retrained softmax")
    ax.fill_between(x, y - sd, y + sd, alpha=0.2)
    ax.axhline(1, color="black", linestyle=":")
    ax.set(title=objective, xlabel="Additional installation units (millions)", ylabel="Ratio to reconstructed status quo")
axes.flat[0].legend()
fig.suptitle("Primary seed 0: mean and one deployment SD across 100 nested paths")
fig.savefig(plots / "nested_deployment.png", dpi=180)
plt.show()
plt.close(fig)


In [ ]:
scores = network_scores(softmax_models[0], experiment["inputs"])
a = softmax_allocation(scores, experiment["capacity"], 2_000_000, temperature, np.random.default_rng(12345))
b = softmax_allocation(scores, experiment["capacity"], 2_000_000, temperature, np.random.default_rng(12345))
assert np.array_equal(a, b)
assert a.sum() == 2_000_000 and (a >= 0).all() and (a <= experiment["capacity"]).all()
assert np.issubdtype(a.dtype, np.integer)
assert np.allclose(scores, np.array([softmax_models[0].activate(x)[0] for x in experiment["inputs"]]), atol=1e-14, rtol=0)
print("Repeatability, integer counts, exact budget, capacities and inference agreement passed; nested paths checked above.")


## Results and interpretation

At two million additional installation units, held-out means for primary evolutionary seed 0 are:

| Policy | Carbon | Energy | Racial equity | Income equity | ZIPs used |
|---|---:|---:|---:|---:|---:|
| Q4 greedy | 1.2880 | 1.1944 | 1.0218 | 1.3901 | 289 |
| Q4 network + softmax | 1.1250 | 0.9731 | 1.1155 | 1.4272 | 10,210.1 |
| Retrained softmax EVA | 1.5533 | 0.9422 | 1.1257 | 1.5282 | 3,508.1 |

All four objective columns are ratios to the reconstructed status quo, so one is the baseline and larger is better. Fractional ZIP counts are averages across draws.

Changing only placement disperses the Q4 seed-0 network across almost every ZIP, lowering both physical-indicator ratios while improving the two equity ratios. Retraining changes score gaps and produces a more concentrated stochastic policy. It increases carbon and both equity ratios relative to greedy Q4 for this seed, but energy is 5.78% below baseline. It does not dominate Q4 or the baseline across all objectives.

| Evolutionary seed | Retrained carbon | Energy | Racial equity | Income equity | Tuning stages | Training criterion met |
|---|---:|---:|---:|---:|---:|---|
| 0 | 1.5533 | 0.9422 | 1.1257 | 1.5282 | 6 | No |
| 1 | 1.5951 | 1.0199 | 1.0852 | 1.4940 | 1 | Yes |
| 2 | 1.4450 | 1.0006 | 1.0567 | 1.4926 | 2 | Yes |

Seed 0 selects stage 2 (the best estimated CPR), not the final failed stage. Seeds 1 and 2 achieve all four baseline ratios in held-out means at the final budget; that does not establish all-budget or every-realization dominance. For retrained seed 0, deployment SDs are approximately 0.000063, 0.000067, 0.000550 and 0.000730 in the four columns. Variation across learned networks is much larger here. Three evolutionary seeds are insufficient to establish general superiority.

The fixed Q4 seed-0 temperature sweep reinforces the calibration issue. At $T=10^{-10}$ it matches greedy allocation in this dataset. At $T=10^{-4}$ it uses about 9,058 ZIPs, with ratios approximately $(1.1951,1.0009,1.0461,1.5447)$. At $T=1$ it uses all 10,243 ZIPs and yields approximately $(1.1000,0.9658,1.0971,1.4089)$. Equity is not monotonic in temperature. Geographic dispersion alone is not a demographic fairness guarantee.

Softmax is a valid controllable stochastic alternative, and retraining is more coherent than assuming ordinal greedy scores are already calibrated probabilities. These experiments show tradeoffs and sensitivity, rather than evidence that softmax is inherently better. The fixed temperature does not guarantee fixed entropy across networks, and the small reused training seed pool may be overfit. A later experiment could compare score-scale controls or entropy targets using a separate validation set; those alternatives are not silently substituted for the raw-score policy requested here.

## Sources and reproducibility

The framework is grounded in the supplied *Multi-objective Evolutionary Learning for Near Pareto-Optimal Optimization of Solar Deployment* (DOI: 10.1145/3736425.3770115), with the siting/equity context supplied by *Lost in Siting: The Hidden Carbon Cost of Inequitable Residential Solar Installations*. The present stochastic rule, implementation and experiments are our extension, not reported results from those papers. Q4 reconstruction limitations continue to apply.

- Pinned source: https://github.com/coopersigrist/SunSight/tree/99588d1696c980075a19abf087c669ae5aa22e4e
- NumPy multinomial definition: https://numpy.org/doc/stable/reference/random/generated/numpy.random.Generator.multinomial.html
- Stable softmax documentation: https://docs.scipy.org/doc/scipy/reference/generated/scipy.special.softmax.html

The sampler was compared with exact enumeration on a small capacity-limited example using 50,000 trials (maximum absolute state-frequency discrepancy 0.00273). Additional checks cover exact budgets, integer counts, capacities, repeatability, native-network inference agreement and nested nonrelocating deployment paths. The deterministic training branch reproduces Q4 seed 0's network scores exactly. These checks validate implementation behavior, not the empirical truth of the data or numerical replication of the original paper.
